# ML Modeling 

Baseline focus:
- `label_sensitive_binary`
- `label_target_main`

Split settings:
- `random` for in-sample capability
- `state` for cross-state generalization


## 1) Imports


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC

import yaml

## 2) Paths and Config


In [ ]:
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CFG_PATH = REPO_ROOT / 'config' / 'paths.local.yaml'
if not CFG_PATH.exists():
    CFG_PATH = REPO_ROOT / 'config' / 'paths.template.yaml'
if not CFG_PATH.exists():
    raise FileNotFoundError('Missing config/paths.local.yaml and config/paths.template.yaml')

cfg = yaml.safe_load(CFG_PATH.read_text(encoding='utf-8')) or {}
shared_cfg = cfg.get('shared', {})
repo_cfg = cfg.get('repo', {})

modeling_input = shared_cfg.get('modeling_input')
if not modeling_input:
    raise ValueError('Set shared.modeling_input in YAML')

DATA_PATH = Path(modeling_input)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUNS_BASE_DIR = REPO_ROOT / repo_cfg.get('modeling_runs_dir', 'results/runs/modeling')
RUN_DIR = RUNS_BASE_DIR / f'run_{RUN_ID}'

print('Config:', CFG_PATH)
print('Dataset:', DATA_PATH)
print('Run dir:', RUN_DIR)

## 3) Run Settings


In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_ROWS = 2_000_000  # None = full dataset. Set int for smoke test.

SPLIT_MODES = ['random', 'state']

STATE_N_SPLITS = 5
STATE_MAX_TRAIN_ROWS = 5_000_000  # Caps train size per fold so large models stay tractable.

DROP_UNKNOWN_TARGET = True  # Drop 'unknown' from label_target_main — labeling artifact, not a real class.

# Max vendor categories kept as distinct OHE columns; rarer vendors are merged into 'infrequent_category'.
# See Feature Preview (§4b) for recommended threshold if running for the first time.
OHE_MAX_CATEGORIES = 50

_BASE_EXCLUDE = {'label_target_main', 'label_sensitive_binary', 'label_sensitive_subtype', 'add_state'}

## 3b) Model Registry

In [ ]:
# Add new models here; they become available to MODELS_TO_RUN without touching run logic.
MODEL_REGISTRY = {
    'logreg': lambda: LogisticRegression(
        solver='saga', max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE,
    ),
    'sgd': lambda: SGDClassifier(
        loss='log_loss', max_iter=1000, tol=1e-4,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=5,
        learning_rate='optimal', class_weight='balanced',
        random_state=RANDOM_STATE, shuffle=True, n_jobs=-1,
    ),
    'rf': lambda: RandomForestClassifier(
        n_estimators=300, max_samples=0.5, random_state=RANDOM_STATE,
        class_weight='balanced_subsample', n_jobs=-1,
    ),
    'lsvc': lambda: CalibratedClassifierCV(
        LinearSVC(max_iter=2000, class_weight='balanced'),
    ),
}

## 3c) Model Selection

In [ ]:
# Edit this list to choose models from MODEL_REGISTRY.
MODELS_TO_RUN = ['logreg']
# e.g. MODELS_TO_RUN = ['logreg', 'sgd']

## 4) Load Dataset + Quick Checkpoints


In [ ]:
if DATA_PATH.suffix == '.parquet':
    pl_df = pl.read_parquet(str(DATA_PATH))
else:
    pl_df = pl.read_csv(str(DATA_PATH), separator='\t')

if MAX_ROWS is not None:
    pl_df = pl_df.sample(n=min(MAX_ROWS, pl_df.height), seed=RANDOM_STATE)

pl_df = pl_df.with_columns(pl.col('add_state').cast(pl.String).str.to_uppercase())

print('Shape:', pl_df.shape)
print(pl_df.head(3))

pdf = pl_df.to_pandas()
del pl_df

## 4b) Feature Preview


In [ ]:
print(f'All dataset columns ({len(pdf.columns)}): {list(pdf.columns)}')
print(f'Always excluded: {sorted(_BASE_EXCLUDE)}\n')

vc = pdf['vendor_clean'].value_counts()
ohe_width = min(len(vc), OHE_MAX_CATEGORIES)
coverage  = vc.head(OHE_MAX_CATEGORIES).sum() / vc.sum()

target_coverage = 0.95
n_for_target = (vc.cumsum() / vc.sum() <= target_coverage).sum() + 1

print(f'vendor_clean: {len(vc)} unique → {ohe_width} OHE cols')
print(f'  top-{OHE_MAX_CATEGORIES} coverage: {coverage:.1%} of rows')
print(f'  vendors needed for {target_coverage:.0%} coverage: {n_for_target}  (current OHE_MAX_CATEGORIES={OHE_MAX_CATEGORIES})')
print(f'\nModels to run: {MODELS_TO_RUN}')

## 5) Modeling Helpers

### Preprocessing
- **`OneHotEncoder`**: expands `vendor_clean` into one 0/1 column per vendor; rare vendors (beyond `OHE_MAX_CATEGORIES`) bundled into `infrequent_category`; `sparse_output=True` keeps the matrix sparse through to the classifier

### Split strategies
- **`random`**: stratified 80/20 — class ratios preserved; tests in-sample capability
- **`state`**: `StratifiedGroupKFold` — all records from the same state stay together; test folds contain only unseen states; 5 folds = train on 4/5 of states, test on 1/5; tests geographic generalization

### Helpers
- **`build_split_folds`**: returns `[(fold_id, train_idx, test_idx)]` for the chosen split strategy
- **`evaluate_model`**: accuracy, macro/weighted F1, confusion matrix, classification report; adds PR-AUC + ROC-AUC for binary targets
- **`run_for_target`**: runs all split modes × folds × `MODELS_TO_RUN` for a single target column; returns list of result dicts

In [ ]:
def build_split_folds(y, split_mode, state_series=None):
    """Return list of (fold_id, train_idx, test_idx) for the given split strategy."""

    if split_mode == 'random':
        train_idx, test_idx = train_test_split(
            y.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
        )
        return [('random_fold_1', pd.Index(train_idx), pd.Index(test_idx))]

    if split_mode == 'state':
        groups = state_series.astype('string').str.upper().str.strip()
        valid  = groups.notna() & (groups != '')
        y_v    = y.loc[valid]
        g_v    = groups.loc[valid]

        n_groups = int(g_v.nunique())
        if n_groups < 3:
            raise ValueError(f'Need >=3 states for grouped CV, got {n_groups}')

        n_splits = min(STATE_N_SPLITS, n_groups)
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        folds = []
        idx = y_v.index
        for i, (tr, te) in enumerate(splitter.split(np.zeros((len(y_v), 1)), y_v, g_v), start=1):
            folds.append((f'state_fold_{i}', pd.Index(idx[tr]), pd.Index(idx[te])))

        print(f'  State split: {n_groups} states → {n_splits} folds')
        return folds

    raise ValueError(f'Unknown split_mode: {split_mode}')

In [ ]:
def evaluate_model(pipe, X_test, y_test):
    """Compute classification metrics. For binary targets also computes PR-AUC and ROC-AUC."""
    pred = pipe.predict(X_test)
    out = {
        'accuracy':    float(accuracy_score(y_test, pred)),
        # macro_f1: equal weight per class — fair to minority classes.
        'macro_f1':    float(f1_score(y_test, pred, average='macro',    zero_division=0)),
        # weighted_f1: weighted by class size — dominated by majority class.
        'weighted_f1': float(f1_score(y_test, pred, average='weighted', zero_division=0)),
        'confusion_matrix':      confusion_matrix(y_test, pred),
        'classification_report': classification_report(y_test, pred, output_dict=True, zero_division=0),
    }
    # PR-AUC is more informative than ROC-AUC under class imbalance.
    if len(pd.Series(y_test).dropna().unique()) == 2 and hasattr(pipe, 'predict_proba'):
        proba = pipe.predict_proba(X_test)[:, 1]
        out['pr_auc']  = float(average_precision_score(y_test, proba))
        out['roc_auc'] = float(roc_auc_score(y_test, proba))
    return out


In [ ]:
def run_for_target(target_col, df):
    """Run all split modes x folds x MODELS_TO_RUN for one target column."""
    work_df = df[df[target_col].notna()].copy()

    if target_col == 'label_target_main' and DROP_UNKNOWN_TARGET:
        before  = len(work_df)
        work_df = work_df[work_df[target_col] != 'unknown'].copy()
        print(f'  Dropped {before - len(work_df):,} "unknown" rows')

    y_full       = work_df[target_col]
    state_series = work_df['add_state']

    print(f'\nTarget={target_col}, n={len(work_df):,}, models={MODELS_TO_RUN}')

    results = []
    for split_mode in SPLIT_MODES:
        folds = build_split_folds(y_full, split_mode=split_mode, state_series=state_series)

        for fold_id, train_idx, test_idx in folds:
            if split_mode == 'state' and STATE_MAX_TRAIN_ROWS and len(train_idx) > STATE_MAX_TRAIN_ROWS:
                rng       = np.random.default_rng(RANDOM_STATE)
                train_idx = pd.Index(rng.choice(train_idx, size=STATE_MAX_TRAIN_ROWS, replace=False))

            X_train = work_df.loc[train_idx, ['vendor_clean']]
            X_test  = work_df.loc[test_idx,  ['vendor_clean']]
            y_train = y_full.loc[train_idx]
            y_test  = y_full.loc[test_idx]

            for model_name in MODELS_TO_RUN:
                pipe = Pipeline([
                    ('ohe', OneHotEncoder(
                        handle_unknown='infrequent_if_exist',
                        max_categories=OHE_MAX_CATEGORIES,
                        sparse_output=True,
                    )),
                    ('classifier', MODEL_REGISTRY[model_name]()),
                ])
                pipe.fit(X_train, y_train)
                metric = evaluate_model(pipe, X_test, y_test)

                out = {
                    'target': target_col, 'split_mode': split_mode,
                    'fold_id': fold_id,   'model': model_name,
                    'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
                    **metric,
                }
                results.append(out)
                print(f'  {target_col} | {split_mode} | {fold_id} | {model_name} → macro_f1={out["macro_f1"]:.4f}')

    return results

## 6a) Run → `label_target_main`

Run this cell independently to train and evaluate on the land-use target only.


In [ ]:
results_main = run_for_target('label_target_main', pdf)
print('Total runs (label_target_main):', len(results_main))


## 6b) Run → `label_sensitive_binary`

Run this cell independently to train and evaluate on the sensitive/non-sensitive binary target only.


In [ ]:
results_binary = run_for_target('label_sensitive_binary', pdf)
print('Total runs (label_sensitive_binary):', len(results_binary))


## 7) Summary + Save

In [ ]:
# Combine whichever result lists were populated — safe if only one target cell was run.
all_results = [
    *globals().get('results_main',   []),
    *globals().get('results_binary', []),
]

summary = pd.DataFrame([
    {
        'target':      r['target'],
        'split_mode':  r['split_mode'],
        'fold_id':     r.get('fold_id', 'na'),
        'model':       r['model'],
        'n_train':     r['n_train'],
        'n_test':      r['n_test'],
        'accuracy':    r['accuracy'],
        'macro_f1':    r['macro_f1'],
        'weighted_f1': r['weighted_f1'],
        'pr_auc':      r.get('pr_auc',  float('nan')),
        'roc_auc':     r.get('roc_auc', float('nan')),
    }
    for r in all_results
]).sort_values(['target', 'split_mode', 'model', 'fold_id'])

RUN_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(RUN_DIR / 'summary_metrics.csv', index=False)
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(
    {'run_id': RUN_ID, 'data_path': str(DATA_PATH), 'models_run': MODELS_TO_RUN},
    indent=2,
))
print('Saved to:', RUN_DIR)
display(summary)

## 8) Detailed Metrics (Optional)

In [ ]:
for r in all_results:
    label = f"{r['target']}  ·  {r['split_mode']}  ·  {r['fold_id']}  ·  {r['model']}"
    print(f'\n{"─" * 72}\n{label}\n')

    classes = [k for k in r['classification_report'] if k not in ('accuracy', 'macro avg', 'weighted avg')]
    cm = pd.DataFrame(r['confusion_matrix'], index=classes, columns=classes)
    cm.index.name   = 'actual \\ pred'
    display(cm)

    print()
    display(pd.DataFrame(r['classification_report']).T.round(3))